<a href="https://colab.research.google.com/github/kaganakman/NextTokenPrediction/blob/main/NextTokenPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

from dataclasses import dataclass
from functools import partial
from typing import Any

In [ ]:
# The below decorator can be swapped with @jax.jit(static_argnames=['ell'])
# for more recent jax versions, which are not available out of the box in
# colab apperantly. Not a huge deal as it can be easily updated.
@partial(jax.jit, static_argnames=['ell'])
def reznik(probs: jax.Array, ell: int) -> jax.Array:
    """
        Python/JAX port of my previous implementation of Reznik's algorithm.
    """
    ell_float = float(ell)
    scaled_probs = ell_float * probs

    k_primes = jnp.floor(scaled_probs + 0.5)
    n_prime = jnp.sum(k_primes)

    delta = k_primes - scaled_probs
    sorted_indices = jnp.argsort(delta)
    ranks = jnp.argsort(sorted_indices)

    delta_val = n_prime - ell
    delta_int = delta_val.astype(jnp.int32)
    abs_delta = jnp.abs(delta_int)

    length = probs.shape[0]

    mask_pos = ranks >= (length - abs_delta)
    mask_pos_int = mask_pos.astype(float)

    mask_neg = ranks < abs_delta
    mask_neg_int = mask_neg.astype(float)

    is_positive = delta_val > 0
    is_positive_float = is_positive.astype(float)

    is_negative = delta_val < 0
    is_negative_float = is_negative.astype(float)

    subtraction_term = mask_pos_int * is_positive_float
    addition_term = mask_neg_int * is_negative_float

    k_primes_adjusted = k_primes - subtraction_term
    k_primes_final = k_primes_adjusted + addition_term

    quantised_probs = k_primes_final / ell_float

    return quantised_probs

batched_reznik = jax.vmap(reznik)

@jax.jit
def Q_n(s: Any, S_n: jax.Array) -> Any:
    """
        This returns the closest element to s in the quantizer set S_n.
        We take jnp.argmin(diffs)[0] to get one element only in case
        there are more.
    """
    diffs = jnp.abs(s - S_n)
    return S_n[np.argmin(diffs)[0]]

In [ ]:
@jax.jit
def cross_entropy(p, q) -> float:
    # note the cost is defined for P(S_n) x P(S_n)
    # p,q have shape mu_i
    q = jnp.clip(q, 1e-10, 1)
    return -jnp.sum(p * jnp.log(q))


In [ ]:
@jax.tree_util.register_dataclass
@dataclass
class Action:
    W: jax.Array
    A: jax.Array
    b: jax.Array
    Q: jax.Array
    K: jax.Array
    V: jax.Array

In [ ]:
@jax.jit
def dirac(s: Any) -> jax.Array:
    mu_i = jnp.zeros(n)
    idx = jnp.where(S_n == s)[0][0]
    new_mu_i = mu_i.at[idx].set(1.0)
    return new_mu_i

@jax.jit
def measure_to_state(mu: jax.Array) -> Any:  #works for any shape of mu (mu_i/mu_k/mu_t)
    #this returns that with the highest probability. we may want to impliment this to return all matches in the future
    return S_n[jnp.argmax(mu, axis = -1)]

batched_measure_to_state = jax.vmap(measure_to_state)

@jax.jit
def one_hot(x: int, n: int) -> jax.Array:
    v = jnp.zeros(n)
    new_v = v.at[x].set(1.0)
    return new_v

@jax.jit
def ensemble_to_index(mu_t: jax.Array) -> int:
    state = jnp.argmax(mu_t, axis = -1).flatten() #look at the S_n axis
    index = sum(int(state[i]) * (n ** i) for i in range(N * len(mu_t)))
    return index

In [ ]:
@jax.jit
def f(s: Any, mu_k: jax.Array, u: Action) -> Any:

    ff = u.W @ jax.nn.relu(u.A @ s + u.b)

    def attention_score(mu_i: jax.Array) -> float:
        return s @ u.Q @ measure_to_state(mu_i) @ u.K

    scores = jax.vmap(attention_score)(mu_k)

    weights = jax.nn.softmax(beta * scores)
    attn = jnp.sum(weights * u.V * batched_measure_to_state(mu_k))

    return jnp.clip(attn + ff, *S)

@jax.jit
def phi_n(u: Action, mu_k: jax.Array) -> jax.Array:

    def process_measure(mu_i: jax.Array) -> jax.Array:
        s = measure_to_state(mu_i)
        new_s = Q_n(f(s, mu_k, u), S_n)

        return dirac(new_s)

    return jax.vmap(process_measure)(mu_k)

batched_phi_n = jax.vmap(phi_n, in_axes=(None, 0))

@jax.jit
def phi(u: Action, mu_t: jax.Array) -> jax.Array:
    next_unquantized_measures = batched_phi_n(u, mu_t)
    result = batched_reznik(next_unquantized_measures)
    return result

In [ ]:
@jax.jit
def C_T(mu_T: jax.Array, y: jax.Array) -> float:
    terminal_mu = mu_T[:, -1]

    batched_losses = jax.vmap(cross_entropy)(terminal_mu, y)

    return jnp.mean(batched_losses)

def create_reachable_ensembles(mu_0, target_depth = None):
    #breadth first search with capped depth of T
    count = 0
    visited = {}
    queue = []
    #keep track of ensemble, depth

    start_ens = mu_0
    start_idx = ensemble_to_index(start_ens)

    visited[0] = {start_idx}
    queue.append((start_ens, 0))

    head = 0

    while head < len(queue):
        curr_ens, depth = queue[head]
        head += 1

        if target_depth is None or depth == target_depth:
            yield curr_ens

        if depth < T:
            if depth + 1 not in visited:
                visited[depth + 1] = set()
            for u in create_actions(U_m):
                next_ens = phi(u, curr_ens)
                next_idx = ensemble_to_index(next_ens)
                count += 1
                if next_idx not in visited[depth + 1]:
                    visited[depth + 1].add(next_idx)
                    queue.append((next_ens, depth + 1))



In [ ]:
# BEEFY FUNCTIONS

def generate_process(n, K_tilde, N, time_horizon):
    pass

In [ ]:
def create_pairs(X, S_n, N, K_tilde):
    x = []
    y_tilde = []

    #start at index N
    for r in range(K_tilde - N):
        x.append(S_n[X[r:r+N]])
        y_tilde.append(S_n[X[r + N]])

    #D_tilde = list(zip(x, y_tilde))
    return x, y_tilde

def construct_empirical_distribution(x, y_tilde, N, S_n):
    #create empirical distributions, remove duplicates
    K_tilde = len(x)
    mu_0 = []
    y = []

    for k in range(K_tilde):

        candidate_x = np.array([dirac(Q_n(x[k][i], S_n)) for i in range(N)])
        candidate_y = dirac(Q_n(y_tilde[k], S_n))

        if k > 0:
            matches = np.all(np.all(np.array(mu_0) == candidate_x, axis = 2), axis=1)
        else:
            matches = []

        if np.any(matches):
            idx = np.where(matches == True)[0][0] # we choose the first occurance
            y[idx] += candidate_y

        else:
            y.append(candidate_y)
            mu_0.append(candidate_x)

    for k in range(len(y)):
        denom = np.sum(y[k])
        y[k] /= denom

    return np.array(mu_0), y

In [ ]:
# VISUALIZATION

def visualize(mu_arr, y_labels):
    K_viz = len(y_labels)
    before = measure_to_state(mu_arr[0])[:, -1]
    after = measure_to_state(mu_arr[T])[:, -1]
    labels = measure_to_state(np.array(y_labels))

    loss_before = C_T(mu_arr[0], y_labels, LOSS)
    loss_after = C_T(mu_arr[T], y_labels, LOSS)

    plt.plot(labels, 'o', label='label')
    plt.plot(before, 's', label=f'before (loss = {loss_before:.2f})', color='gray')
    plt.plot(after, 's', label=f'after (loss = {loss_after:.2f})', color='orange')

    for k in range(K_viz):
        plt.plot([k-0.02, k-0.02], [before[k], labels[k]], '-', color='gray', lw=1)
        plt.plot([k+0.02, k+0.02], [after[k], labels[k]], '-', color='orange', lw=1)

    plt.legend()
    plt.show()

    plt.fill_between(np.arange(K_viz), before - labels, alpha=0.25, label=f'before (loss = {loss_before:.2f})', color='gray')
    plt.fill_between(np.arange(K_viz), after - labels, alpha=0.25, label=f'after (loss = {loss_after:.2f})', color='orange')

    for k in range(K_viz):
        plt.plot([k-0.02, k-0.02], [(before - labels)[k], 0], color='gray', lw=1)
        plt.plot([k+0.02, k+0.02], [(after - labels)[k], 0], color='orange', lw=1)

    plt.legend()
    plt.show()

In [ ]:
# PARAMETERS

T = 2       # time horizon = number of layers

l = 9       # probability measure quantizations

n = 2       # state space quantizations

m = 3       # action space quantization

N = 2       # length of prompt/order of markov chain

K_tilde = 98 #number of training pairs (state labels)

xi = 0.8    # 0 = non-Markovian, 1 = Markovian

beta = 0.3  # attention temperature

LOSS = 'Cross Entropy'

# S is the interval [0, 5]
S = (-2.5, 2.5)
#d = 1

#create example quantized state space
S_n = np.linspace(*S, n)

#used for W2
cost_matrix = np.array([[np.linalg.norm(s1 - s2)**2 for s1 in S_n] for s2 in S_n])

P_l = {i/(l -1) for i in range(0, l)}
U_m = Action((1, 1), (-2, 2), (-2 , 2), (1,1), (-2,2), (-2, 2), m = m)



In [ ]:
def train(xi):
    #construct dataset
    X = generate_process(n, K_tilde,  N , xi)
    x, y_tilde = create_pairs(X, S_n, N, K_tilde)

    #lift and dedup the dataset
    mu_0, y = construct_empirical_distribution(x, y_tilde, N, S_n)
    mu = np.zeros((T + 1, *mu_0.shape))
    mu[0] = mu_0

    #define terminal cost
    C = {}
    for t in range(T+1):
        C[t] = {}
    for i, mu_T in enumerate(create_reachable_ensembles(mu[0], target_depth=T)):
        C[T][ensemble_to_index(mu_T)] = C_T(mu_T, y, LOSS)

    gamma = {}
    for t in range(T):
        gamma[t] = {}
    actions = list(create_actions(U_m))

    #solve dp
    for t in range(T-1, -1, -1): #goes from T-1 to 0
        for mu_t in create_reachable_ensembles(mu[0], target_depth=t):

            i = ensemble_to_index(mu_t)
            costs = [C[t+1][ensemble_to_index(phi(u, mu_t))] for u in actions] #costs indexed by each action
            best_idx = np.argmin(costs)
            gamma[t][i] = actions[best_idx]
            C[t][i] = np.min(costs)

    #forward pass
    U_t = []
    for t in range(T):
        optimal_u = gamma[t][ensemble_to_index(mu[t])]
        mu[t + 1] = phi(optimal_u, mu[t])
        U_t.append(optimal_u)

    return U_t, mu, y


# TESTING
def test(U_t, xi):
    X_test = generate_process(n, K_tilde, N, xi=0.5)
    x_test, y_tilde_test = create_pairs(X_test, S_n, N, K_tilde)
    mu_0_test, y_test = construct_empirical_distribution(x_test, y_tilde_test, N, S_n)

    mu_0_test = np.array(mu_0_test)
    mu_test = np.zeros((T + 1, *mu_0_test.shape))
    mu_test[0] = mu_0_test

    #Forward pass
    for t in range(T):
        mu_test[t + 1] = phi(U_t[t], mu_test[t])

    return mu_test, y_test

if __name__ == "__main__":
    for xi in {0.01, 0.5, 1.0}:
        actions, mu, y = train(xi)
        visualize(mu, y)
        test(actions, xi)


